In [ ]:
# 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import joblib
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve, make_scorer, precision_score, recall_score, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

In [ ]:
# 2. Data Loading

In [ ]:
train_df = pd.read_csv("UNSW_NB15_training-set.csv")
test_df  = pd.read_csv("UNSW_NB15_testing-set.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())

In [ ]:
# 3. Data Cleaning and Feature Engineering

In [ ]:
def clean_dataset(df):
    df = df.copy()
    
    # Standardise proto column
    if "proto" in df.columns:
        df["proto"] = df["proto"].astype(str).str.lower()
    
    return df

train_df = clean_dataset(train_df)
test_df  = clean_dataset(test_df)

In [ ]:
def add_engineered_features(df):
    df = df.copy()

    # Safe numeric conversions if columns exist
    needed_cols = ["sbytes", "dbytes", "spkts", "dpkts", "dur"]
    for col in needed_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Total traffic features
    if {"sbytes", "dbytes"}.issubset(df.columns):
        df["total_bytes"] = df["sbytes"] + df["dbytes"]

    if {"spkts", "dpkts"}.issubset(df.columns):
        df["total_pkts"] = df["spkts"] + df["dpkts"]

    # Ratio features
    if {"sbytes", "dbytes"}.issubset(df.columns):
        df["bytes_ratio"] = df["sbytes"] / (df["dbytes"] + 1)

    if {"spkts", "dpkts"}.issubset(df.columns):
        df["pkt_ratio"] = df["spkts"] / (df["dpkts"] + 1)

    # Rate / intensity features
    if {"sbytes", "dbytes", "dur"}.issubset(df.columns):
        df["bytes_per_sec"] = (df["sbytes"] + df["dbytes"]) / (df["dur"] + 1)

    if {"spkts", "dpkts", "dur"}.issubset(df.columns):
        df["pkts_per_sec"] = (df["spkts"] + df["dpkts"]) / (df["dur"] + 1)

    # Replace infinities created by division
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

    return df

In [ ]:
# 4. Initial Dataset Inspection

In [ ]:
# Quick schema check
print("Columns:", len(train_df.columns))
display(pd.DataFrame({
    "column": train_df.columns,
    "dtype": train_df.dtypes.astype(str),
}).head(30))

In [ ]:
# Check for common target columns in UNSW-NB15
possible_targets = [c for c in ["label", "attack_cat", "class", "target", "y"] if c in train_df.columns]
print("Possible target columns found:", possible_targets)

In [ ]:
# 5. Data Analysis

In [ ]:
# Missing values summary
missing = train_df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:", len(missing))
display(missing.head(30))

In [ ]:
# Check for infinite values 
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(train_df[numeric_cols]).sum().sort_values(ascending=False)
inf_counts = inf_counts[inf_counts > 0]
print("Numeric columns with inf:", len(inf_counts))
display(inf_counts.head(30))

In [ ]:
# Class distribution (works once we know the target column)
def show_target_distribution(df, target_col: str):
    vc = df[target_col].value_counts(dropna=False)
    dist = (vc / len(df) * 100).round(2)
    out = pd.DataFrame({"count": vc, "percent": dist})
    display(out)

# If you already know your target, set it here:
TARGET_COL = "label"  # e.g. "label" for binary, or "attack_cat" for multiclass

if TARGET_COL is not None and TARGET_COL in train_df.columns:
    show_target_distribution(train_df, TARGET_COL)
else:
    print("Set TARGET_COL to view class distribution (e.g. 'label' or 'attack_cat').")

In [ ]:
# Count values in the target column
counts = train_df['label'].value_counts()

# Create bar chart
plt.figure()
counts.plot(kind='bar')

# Labels and title
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Class Distribution (Normal vs Attack)")

plt.xticks([0,1], ["Normal", "Attack"], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(train_df["dur"], bins=50, log_scale=True)

plt.title("Distribution of Flow Duration (Log Scale)")
plt.xlabel("Flow Duration")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=train_df["label"], y=train_df["dur"])
plt.yscale("log")
plt.xlabel("Class")
plt.ylabel("Flow Duration")
plt.title("Flow Duration by Class")
plt.xticks([0, 1], ["Normal", "Attack"])
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
corr = train_df.corr(numeric_only=True)

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
if "attack_cat" in train_df.columns:
    attack_counts = train_df["attack_cat"].value_counts()

    plt.figure()
    attack_counts.plot(kind="bar")
    plt.xlabel("Attack Type")
    plt.ylabel("Count")
    plt.title("Distribution of Attack Categories")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("'attack_cat' column not found.")

In [ ]:
# 6. Train/ Test Feature Setup

In [ ]:
TARGET_COL = "label"  # Binary classification target

# Ensure target exists BEFORE doing anything
assert TARGET_COL in train_df.columns, f"TARGET_COL '{TARGET_COL}' not found in dataset columns."

# Build drop list (ID + leakage protection)
DROP_COLS = []

if "id" in train_df.columns:
    DROP_COLS.append("id")

# Prevent label leakage
if TARGET_COL == "label" and "attack_cat" in train_df.columns:
    DROP_COLS.append("attack_cat")

print("Dropping columns:", DROP_COLS)

# Create features and labels
X = train_df.drop(columns=[TARGET_COL] + DROP_COLS)
y = train_df[TARGET_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
TARGET_COL = "label"

DROP_COLS = []

if "id" in train_df.columns:
    DROP_COLS.append("id")

# Drop attack_cat for binary classification to prevent leakage
if TARGET_COL == "label" and "attack_cat" in train_df.columns:
    DROP_COLS.append("attack_cat")

X_train = train_df.drop(columns=DROP_COLS + [TARGET_COL])
y_train = train_df[TARGET_COL]

X_test = test_df.drop(columns=DROP_COLS + [TARGET_COL])
y_test = test_df[TARGET_COL]

# Ensure train and test have the same feature columns
assert list(X_train.columns) == list(X_test.columns), "Train/test column mismatch detected."

print("Dropping columns:", DROP_COLS)
print("Train and test columns match.")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
TARGET_COL = "label"

DROP_COLS = []

if "id" in train_df.columns:
    DROP_COLS.append("id")

# Drop attack_cat for binary classification to prevent leakage
if TARGET_COL == "label" and "attack_cat" in train_df.columns:
    DROP_COLS.append("attack_cat")

X_train = train_df.drop(columns=DROP_COLS + [TARGET_COL])
y_train = train_df[TARGET_COL]

X_test = test_df.drop(columns=DROP_COLS + [TARGET_COL])
y_test = test_df[TARGET_COL]

# Ensure train and test have the same feature columns
assert list(X_train.columns) == list(X_test.columns), "Train/test column mismatch detected."

print("Dropping columns:", DROP_COLS)
print("Train and test columns match.")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
print("Non-numeric columns in X_train:")
print(X_train.select_dtypes(exclude=[np.number]).columns.tolist())

if "attack_cat" in X_train.columns:
    raise ValueError("attack_cat is still present in X_train and must be removed.")

In [ ]:
# 7. Create Original and Engineered Feauture Sets

In [ ]:
# New features 
# Version A: original features
X_train_orig = X_train.copy()
X_test_orig = X_test.copy()

# Version B: engineered features
train_df_eng = add_engineered_features(train_df)
test_df_eng = add_engineered_features(test_df)

# Keep drop logic aligned with your current setup
DROP_COLS_ENG = []
if "id" in train_df_eng.columns:
    DROP_COLS_ENG.append("id")
if TARGET_COL == "label" and "attack_cat" in train_df_eng.columns:
    DROP_COLS_ENG.append("attack_cat")

X_train_eng = train_df_eng.drop(columns=DROP_COLS_ENG + [TARGET_COL])
X_test_eng = test_df_eng.drop(columns=DROP_COLS_ENG + [TARGET_COL])

print("Original feature count:", X_train_orig.shape[1])
print("Engineered feature count:", X_train_eng.shape[1])
print("New engineered columns:", sorted(set(X_train_eng.columns) - set(X_train_orig.columns)))

In [ ]:
# 8. Identify Numeric and Categorical Features

In [ ]:
candidate_categorical = ["service", "state", "proto"]

# Original feature version
num_cols_orig = X_train_orig.select_dtypes(include=[np.number]).columns
X_train_orig[num_cols_orig] = X_train_orig[num_cols_orig].replace([np.inf, -np.inf], np.nan)
X_test_orig[num_cols_orig] = X_test_orig[num_cols_orig].replace([np.inf, -np.inf], np.nan)

categorical_features_orig = X_train_orig.select_dtypes(exclude=[np.number]).columns.tolist()
numeric_features_orig = X_train_orig.select_dtypes(include=[np.number]).columns.tolist()

print("Original numeric features:", len(numeric_features_orig))
print("Original categorical features:", len(categorical_features_orig))
print("Original categorical columns:", categorical_features_orig)


# Engineered feature version
num_cols_eng = X_train_eng.select_dtypes(include=[np.number]).columns
X_train_eng[num_cols_eng] = X_train_eng[num_cols_eng].replace([np.inf, -np.inf], np.nan)
X_test_eng[num_cols_eng] = X_test_eng[num_cols_eng].replace([np.inf, -np.inf], np.nan)

categorical_features_eng = X_train_eng.select_dtypes(exclude=[np.number]).columns.tolist()
numeric_features_eng = X_train_eng.select_dtypes(include=[np.number]).columns.tolist()

print("Engineered numeric features:", len(numeric_features_eng))
print("Engineered categorical features:", len(categorical_features_eng))
print("Engineered categorical columns:", categorical_features_eng)

In [ ]:
# 9. Build Preprocessing Pipelines

In [ ]:
numeric_transformer_orig = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_orig = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_orig = ColumnTransformer(transformers=[
    ("num", numeric_transformer_orig, numeric_features_orig),
    ("cat", categorical_transformer_orig, categorical_features_orig)
])


numeric_transformer_eng = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_eng = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_eng = ColumnTransformer(transformers=[
    ("num", numeric_transformer_eng, numeric_features_eng),
    ("cat", categorical_transformer_eng, categorical_features_eng)
])
print("Preprocessor built.")